# Day 3: Clustering with K-Means (Palmer Penguins)

This notebook is a first look at unsupervised learning using the K-Means clustering algorithm.

### Motivation

The models you have fit so far were supervised: every observation came with a label or target value, and the model learned to predict it. Clustering is *unsupervised*: there are no labels. The goal is to discover groups of similar observations using only the features.

K-Means is the most widely used clustering algorithm. You choose the number of clusters, `k`, and the algorithm divides the data into `k` groups of similar observations.

**Dataset.** [Palmer Penguins](https://allisonhorst.github.io/palmerpenguins/): 344 penguins from three species (Adelie, Chinstrap, Gentoo) sampled near Palmer Station, Antarctica, with several body measurements per observation. Source: Gorman et al.

**Objective.** Step through the clustering workflow: use K-Means to find 3 clusters in the penguin body measurements, then compare the discovered clusters to the actual species labels, which the algorithm never sees.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


### Load the dataset

`seaborn` ships a copy of Palmer Penguins. We drop rows with missing measurements so every subsequent section operates on the same complete-case dataframe.

In [ ]:
df = sns.load_dataset("penguins").dropna().reset_index(drop=True)
print("Shape after dropping NaN:", df.shape)
df.head()


<details>
<summary><strong>Data dictionary</strong> (click to expand)</summary>

<br>

Source: [palmerpenguins package](https://allisonhorst.github.io/palmerpenguins/).

| Column | Meaning |
| --- | --- |
| `species` | Penguin species: Adelie, Chinstrap, or Gentoo |
| `island` | Island where the penguin was observed (Biscoe, Dream, or Torgersen) |
| `bill_length_mm` | Length of the bill, in millimeters |
| `bill_depth_mm` | Depth of the bill, in millimeters |
| `flipper_length_mm` | Length of the flipper, in millimeters |
| `body_mass_g` | Body mass, in grams |
| `sex` | Male or Female |

</details>

### Summary statistics

The dataframe includes a `species` column, but the clustering will not use it. The algorithm sees only the numeric measurements. We keep `species` in the dataframe so that at the end we can check how well the discovered clusters line up with the actual species.

In [ ]:
print("Species distribution:")
print(df["species"].value_counts())

print("\nNumeric feature summary:")
print(df[["bill_length_mm","bill_depth_mm","flipper_length_mm","body_mass_g"]].describe().round(1))


### Is there group structure in the measurements?

Clustering is useful when observations form groups in feature space. The scatter plot below shows bill length against flipper length with no species information. This is exactly what a clustering algorithm "sees."

Even without labels, the points appear to form distinct groups: one group with long flippers in the upper region, and one or two groups with shorter flippers below it. K-Means gives us a way to assign each point to a group automatically.

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="bill_length_mm", y="flipper_length_mm",
                s=70, alpha=0.8, color="grey")
plt.title("Palmer Penguins: bill length vs flipper length (no labels)")
plt.xlabel("Bill length (mm)")
plt.ylabel("Flipper length (mm)")
plt.show()


## The clustering workflow

The workflow has four steps:

1. Choose the features to cluster on and put them on the same scale.
2. Fit K-Means with a chosen number of clusters `k`.
3. Plot the cluster assignments.
4. Interpret the clusters.

### Step 1: choose features and put them on the same scale

We cluster on two measurements: bill length and flipper length. K-Means groups observations by how close they are to each other in feature space, so the features need to be on comparable scales. Flipper length has larger numbers than bill length (around 200 mm vs around 44 mm), so without rescaling it would count for more just because of its units. `StandardScaler` rescales each feature to mean 0 and standard deviation 1.

> **New scikit-learn pattern.** Preprocessing objects such as `StandardScaler` use `.fit(X)` to learn the rescaling parameters (each column's mean and standard deviation) and `.transform(X)` to apply them. Also note there is no train-test split here: clustering has no labels to predict, so there is nothing to hold out, and we work with the full dataset.

In [ ]:
features = ["bill_length_mm", "flipper_length_mm"]
X = df[features].values

scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)

print("Feature matrix shape:", X_scaled.shape)
print("Column means after scaling:", X_scaled.mean(axis=0).round(2))
print("Column stds after scaling: ", X_scaled.std(axis=0).round(2))


### Step 2: fit K-Means with k = 3

`KMeans` looks for 3 groups of observations that sit close together in the feature space. After fitting, `kmeans.labels_` holds the cluster (0, 1, or 2) assigned to each observation.

The cluster numbers are arbitrary. Rerunning with a different `random_state` may produce the same grouping with the labels 0, 1, and 2 swapped around.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X_scaled)

df["cluster"] = kmeans.labels_

print("Cluster sizes:")
print(df["cluster"].value_counts().sort_index())


### Step 3: plot the cluster assignments

The scatter plot below repeats the unlabeled plot from earlier, now colored by cluster assignment.

In [ ]:
plt.figure(figsize=(7.5, 5.5))
sns.scatterplot(data=df, x="bill_length_mm", y="flipper_length_mm",
                hue="cluster", palette="Set2", s=70, alpha=0.8)
plt.title("K-Means clusters (k = 3)")
plt.xlabel("Bill length (mm)")
plt.ylabel("Flipper length (mm)")
plt.show()


### Step 4: interpret the clusters

What did K-Means find? The algorithm never saw the `species` column, so we can use it as an answer key: a cross-tabulation counts how many penguins of each species landed in each cluster.

In [ ]:
comparison = pd.crosstab(df["cluster"], df["species"])
print(comparison)


**Result.** Each cluster is dominated by a single species: K-Means recovered the species structure from two body measurements alone, without ever seeing a label. Most of the disagreements are Adelie and Chinstrap penguins near the boundary between their groups, where the two species overlap in these two measurements.

One caution: this check was possible only because the penguins dataset happens to include species labels. In a genuine unsupervised problem there is no answer key. The clusters have to be judged by plotting them and by whether they make sense to someone who knows the domain.

### Practice exercises

Complete at least one.

**(a) Vary `k`.** The number of clusters is your choice, and the algorithm will produce exactly as many groups as you ask for. Refit K-Means on `X_scaled` with `k = 2` and `k = 5`, and plot the cluster assignments for each (the scatter plot code above can be reused). Describe what happens: which species end up sharing a cluster at `k = 2`? What does the algorithm do with the extra clusters at `k = 5`? Use `pd.crosstab` to check your reading of the plots.

**(b) Different features.** Recluster with `k = 3` using `flipper_length_mm` and `body_mass_g` instead of bill length (remember to standardize the new feature matrix). Cross-tabulate the new clusters against species. Which species can no longer be separated, and why? Hint: both of these features mainly measure overall body size.

Worked solutions: `day03_kmeans_teacher.ipynb`.

In [ ]:
# Complete at least one of (a) and (b).



## Summary

The clustering workflow:

1. Choose the features to cluster on and standardize them so they are on the same scale.
2. Fit `KMeans(n_clusters=k)` on the scaled features. You choose `k`.
3. Plot the cluster assignments.
4. Interpret the clusters. Here we could check them against the species labels; in a genuine unsupervised problem there is no answer key.

Unlike the supervised workflow, there is no target variable, no train-test split, and no accuracy score.